# Thu thập và lưu snapshot dữ liệu lịch sử thị trường

## Mục tiêu

Notebook này được xây dựng nhằm thu thập dữ liệu lịch sử từ nhiều nhóm tài sản tài chính khác nhau thông qua Yahoo Finance, bao gồm:

- Tiền mã hóa (Cryptocurrency)
- Chỉ số chứng khoán (Stock Index)
- Cổ phiếu (Stocks)
- Hàng hóa (Commodities)
- Ngoại hối (Forex)

Dữ liệu được thu thập với cấu hình:

- **Khoảng thời gian:** 6 tháng gần nhất
- **Tần suất:** 1 giờ (1h)

Sau khi thu thập, dữ liệu sẽ được xử lý và chuẩn hóa theo cùng một cấu trúc gồm:

- Mã tài sản (`symbol`)
- Khung thời gian (`timeframe`)
- Giá mở cửa (`open`)
- Giá cao nhất (`high`)
- Giá thấp nhất (`low`)
- Giá đóng cửa (`close`)
- Khối lượng giao dịch (`volume`)
- Thời gian dạng Unix Timestamp (`time_timestamp`)

Ngoài ra, notebook sẽ thực hiện các bước xử lý dữ liệu bao gồm:

- Kiểm tra và loại bỏ dữ liệu thiếu
- Kiểm tra và loại bỏ các bản ghi trùng lặp
- Chuẩn hóa định dạng thời gian sang Unix Timestamp
- Chuẩn hóa dữ liệu theo cùng một cấu trúc OHLCV
- Gộp toàn bộ dữ liệu từ nhiều nhóm tài sản thành một dataset thống nhất
- Lưu dữ liệu thành file CSV snapshot để phục vụ các bước phân tích tiếp theo, bao gồm:

    - Phân tích khám phá dữ liệu tổng quan (Market Overview EDA)
    - Phân tích chuỗi thời gian (Time Series Analysis)
    - Phân tích và phát hiện bất thường (Anomaly Analysis)
    - Xây dựng đặc trưng cho mô hình dự đoán (Feature Engineering)
    - Huấn luyện và đánh giá mô hình dự báo (Forecasting Models)
    - Tích hợp kết quả vào dashboard giám sát và phân tích dữ liệu

Notebook này đóng vai trò là **lớp snapshot dữ liệu**, giúp đảm bảo quá trình phân tích có thể tái lập và không cần gọi API nhiều lần.

In [16]:
import yfinance as yf
import pandas as pd
from datetime import datetime
from pathlib import Path

symbols = [
    "BTC-USD", "ETH-USD", "SOL-USD", "BNB-USD", "XRP-USD",
    "ADA-USD", "DOGE-USD", "AVAX-USD", "LINK-USD",

    "^GSPC", "^IXIC", "^DJI",

    "AAPL", "MSFT", "NVDA", "TSLA", "GOOGL", "AMZN", "META",

    "GC=F", "SI=F",

    "EURUSD=X", "GBPUSD=X", "USDJPY=X",
    "USDCHF=X", "AUDUSD=X", "USDCAD=X",
]

PERIOD = "6mo"
INTERVAL = "1h"

dfs = []
report = []

for sym in symbols:

    print(f"Loading {sym} ...")

    try:
        data = yf.download(
            sym,
            period=PERIOD,
            interval=INTERVAL,
            progress=False
        )

        if data is None or data.empty:
            report.append((sym, 0, "EMPTY"))
            continue


        data = data.reset_index()


        unix_timestamps = (
            data.iloc[:, 0]
            .values
            .astype("datetime64[s]")
            .astype("int64")
        )


        df = pd.DataFrame({
            "symbol": sym,
            "timeframe": INTERVAL,

            "open": data["Open"].values.flatten(),
            "high": data["High"].values.flatten(),
            "low": data["Low"].values.flatten(),
            "close": data["Close"].values.flatten(),
            "volume": data["Volume"].values.flatten(),

            "time_timestamp": unix_timestamps
        })


        df = (
            df
            .dropna(
                subset=[
                    "open",
                    "high",
                    "low",
                    "close"
                ]
            )
            .drop_duplicates(
                subset=[
                    "symbol",
                    "timeframe",
                    "time_timestamp"
                ]
            )
            .sort_values("time_timestamp")
        )

        count = len(df)
        report.append((sym, count, "OK"))
        dfs.append(df)
    except Exception as e:

        report.append(
            (sym, 0, f"ERROR")
        )

if dfs:
    full_df = pd.concat(
        dfs,
        ignore_index=True
    )
else:
    full_df = pd.DataFrame()

report_df = pd.DataFrame(
    report,
    columns=[
        "symbol",
        "records",
        "status"
    ]
).sort_values(
    "records",
    ascending=False
)


print("\nDATA LOADING REPORT")
print("-" * 60)

display(report_df)


print("\nTOTAL ROWS:", len(full_df))

print(
    "TOTAL SYMBOLS LOADED:",
    full_df["symbol"].nunique()
    if not full_df.empty
    else 0
)

display(full_df.head())

Loading BTC-USD ...
Loading ETH-USD ...
Loading SOL-USD ...
Loading BNB-USD ...
Loading XRP-USD ...
Loading ADA-USD ...
Loading DOGE-USD ...
Loading AVAX-USD ...
Loading LINK-USD ...
Loading ^GSPC ...
Loading ^IXIC ...
Loading ^DJI ...
Loading AAPL ...
Loading MSFT ...
Loading NVDA ...
Loading TSLA ...
Loading GOOGL ...
Loading AMZN ...
Loading META ...
Loading GC=F ...
Loading SI=F ...
Loading EURUSD=X ...
Loading GBPUSD=X ...
Loading USDJPY=X ...
Loading USDCHF=X ...
Loading AUDUSD=X ...
Loading USDCAD=X ...

DATA LOADING REPORT
------------------------------------------------------------


,symbol,records,status
0,BTC-USD,4324,OK
1,ETH-USD,4324,OK
2,SOL-USD,4324,OK
3,BNB-USD,4324,OK
4,XRP-USD,4324,OK
5,ADA-USD,4324,OK
6,DOGE-USD,4324,OK
7,AVAX-USD,4324,OK
8,LINK-USD,4324,OK
25,AUDUSD=X,3056,OK



TOTAL ROWS: 71158
TOTAL SYMBOLS LOADED: 27


,symbol,timeframe,open,high,low,close,volume,time_timestamp
0,BTC-USD,1h,106442.296875,106513.968750,105922.125000,105923.773438,0,1762761600
1,BTC-USD,1h,105946.929688,106545.859375,105946.929688,106386.796875,2375245824,1762765200
2,BTC-USD,1h,106424.656250,106424.656250,105976.296875,106005.289062,0,1762768800
3,BTC-USD,1h,106001.789062,106283.828125,105944.421875,106184.195312,0,1762772400
4,BTC-USD,1h,106184.328125,106184.328125,105883.914062,105925.820312,1035689984,1762776000


In [17]:
snapshot_time = datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)
save_dir = Path("../data/snapshots")
save_dir.mkdir(
    parents=True,
    exist_ok=True
)
csv_path = save_dir / f"market_snapshot_{snapshot_time}.csv"
full_df.to_csv(
    csv_path,
    index=False
)
print(f"\nSAVED TO: {csv_path}")


SAVED TO: ..\data\snapshots\market_snapshot_20260510_150719.csv
